# mT0-small RAG Answer Evaluation

## Purpose

Evaluates answers generated by `07b_mt0_small_gold_rag.ipynb` (bigscience/mt0-small, gold-context fine-tuned, then used in RAG with tfidf/bm25/dense retrieval). This replaces the earlier `mt5_gold` attempt, which failed to learn the task.


## Metrics

Same four metrics as the Qwen evaluation, for direct comparability across generators:
- Exact Match (EM)
- Token F1
- ROUGE-L F1
- Semantic similarity (cosine, `intfloat/multilingual-e5-base` — same model as the dense retriever)

In [38]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
import altair as alt
from sentence_transformers import SentenceTransformer

## Configuration

`GENERATION_DIR` should point at wherever the `07b_mt0_small_gold_rag.ipynb` prediction files were copied locally (from `.../artifacts/seq2seq/mt0_small_gold/generation/` on Drive). Copy them into `data/generation/mt0_small/` in the project for a consistent local layout, matching `qwen_colab/` and `mt5_gold/`.

In [39]:
GENERATOR_NAME = "mt0_small_gold_v2"

RETRIEVERS = ("tfidf", "bm25", "dense")

RAG_TOP_K = 5

# SPLIT "validation" je pusten pre testa i sacuvani su fajlovi csv u evaluation folderu
SPLIT = "test"

PROJECT_ROOT = Path.cwd()

GENERATION_DIR = (
    PROJECT_ROOT
    / "data"
    / "results"
    / "seq2seq"
    / "generation"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "evaluation"
    / "mt0_small"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert GENERATION_DIR.exists(), (
    f"Ne postoji: {GENERATION_DIR}. "
    "Prekopiraj *_predictions.jsonl fajlove iz "
    "07b_mt0_small_gold_rag.ipynb (Drive artifacts/generation/) ovde."
)

print("Generisane predikcije:", GENERATION_DIR)
print("Rezultati evaluacije:", RESULTS_DIR)
print("Split:", SPLIT, "| RAG_TOP_K:", RAG_TOP_K)

Generisane predikcije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/results/seq2seq/generation
Rezultati evaluacije: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/mt0_small
Split: test | RAG_TOP_K: 5


In [40]:
def load_jsonl(path: Path):
    records = []

    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as error:
                raise ValueError(
                    f"Neispravan JSON u redu {line_number}: {path}"
                ) from error

    return records

## Loading Generated Predictions

File naming: `{retriever}_{split}_top{RAG_TOP_K}_predictions.jsonl`. Reference answer field is `gold_answer`, same as the mt5_gold attempt.

In [41]:
required_fields = {
    "question_id",
    "question",
    "gold_answer",
    "retriever",
    "generator",
    "generated_answer",
}

generation_data = {}

for retriever in RETRIEVERS:
    path = GENERATION_DIR / f"{retriever}_{SPLIT}_top{RAG_TOP_K}_predictions.jsonl"

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    for record in records:
        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

    generation_data[retriever] = records
    print(f"{retriever}/{SPLIT}: {len(records)} generisanih odgovora")

if not generation_data:
    raise FileNotFoundError(
        f"Nisu pronađeni generisani odgovori za split '{SPLIT}' ni za jedan retriever."
    )

tfidf/test: 22 generisanih odgovora
bm25/test: 22 generisanih odgovora
dense/test: 22 generisanih odgovora


## Loading Generated Predictions

File naming: `{retriever}_{split}_top{RAG_TOP_K}_predictions.jsonl`. Reference answer field is `gold_answer`, same as the mt5_gold attempt.

In [42]:
required_fields = {
    "question_id",
    "question",
    "gold_answer",
    "retriever",
    "generator",
    "generated_answer",
}

generation_data = {}

for retriever in RETRIEVERS:
    path = GENERATION_DIR / f"{retriever}_{SPLIT}_top{RAG_TOP_K}_predictions.jsonl"

    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    records = load_jsonl(path)

    for record in records:
        missing = required_fields - record.keys()
        if missing:
            raise ValueError(
                f"Pitanje {record.get('question_id')} u {path} nema polja: {sorted(missing)}"
            )

    generation_data[retriever] = records
    print(f"{retriever}/{SPLIT}: {len(records)} generisanih odgovora")

if not generation_data:
    raise FileNotFoundError(
        f"Nisu pronađeni generisani odgovori za split '{SPLIT}' ni za jedan retriever."
    )

tfidf/test: 22 generisanih odgovora
bm25/test: 22 generisanih odgovora
dense/test: 22 generisanih odgovora


## Sanity Check — Real Text, Not Sentinel Tokens

Before computing any metrics, check a few raw generated answers directly. This is the single most important check given the previous mt5_gold failure mode (`<extra_id_0>` outputs).

In [43]:
first_retriever = next(iter(generation_data))

for record in generation_data[first_retriever][:3]:
    print(f"Pitanje: {record['question']}")
    print(f"Referentni: {record['gold_answer']}")
    print(f"Generisani: {record['generated_answer']}")
    print("-" * 80)

Pitanje: Opisati testove kompatibilnosti pri testiranju softvera.
Referentni: Testovi kompatibilnosti proveravaju da softver može pravilno da radi u različitim okruženjima i zajedno sa drugim programima, uređajima ili servisima.
Generisani: Testove kompatibilnosti se proverava da je ulazne vrednosti moguće odrediti odgovarajuće izvršavanja, funkcionalnosti, pouzdanosti, sigurnosti i prenosivosti.
--------------------------------------------------------------------------------
Pitanje: Navesti primere fatalnih posledica pri neispravnom softveru.
Referentni: Fatalne posledice mogu nastati kod kritičnih sistema. Therac-25 je zbog softverske greške doveo do najmanje šest slučajeva predoziranja pacijenata zračenjem, pri čemu su tri pacijenta umrla. Kod sistema Patriot greška u računanju vremena dovela je do promašaja od oko 600 metara, pri čemu raketa nije presretnuta, poginulo je 28 vojnika, a više od 100 je ranjeno. Raketa Ariane 5 se zbog softverske greške uništila 37 sekundi nakon polet

## Text Normalization and Metrics

Identical to `09_qwen_evaluation.ipynb`, for direct comparability across generators.

In [44]:
TOKEN_PATTERN = re.compile(r"[\w]+", re.UNICODE)


def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text or "")
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text: str):
    return TOKEN_PATTERN.findall(normalize_text(text))

In [45]:
def compute_exact_match(prediction: str, reference: str) -> int:
    return int(normalize_text(prediction) == normalize_text(reference))


def compute_token_f1(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    from collections import Counter

    pred_counts = Counter(pred_tokens)
    ref_counts = Counter(ref_tokens)

    overlap = sum(
        min(pred_counts[token], ref_counts[token])
        for token in pred_counts
    )

    if overlap == 0:
        return 0.0

    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [46]:
def longest_common_subsequence_length(a: list, b: list) -> int:
    previous_row = [0] * (len(b) + 1)

    for token_a in a:
        current_row = [0] * (len(b) + 1)

        for j, token_b in enumerate(b, start=1):
            if token_a == token_b:
                current_row[j] = previous_row[j - 1] + 1
            else:
                current_row[j] = max(previous_row[j], current_row[j - 1])

        previous_row = current_row

    return previous_row[-1]


def compute_rouge_l(prediction: str, reference: str) -> float:
    pred_tokens = tokenize(prediction)
    ref_tokens = tokenize(reference)

    if not pred_tokens and not ref_tokens:
        return 1.0

    if not pred_tokens or not ref_tokens:
        return 0.0

    lcs_length = longest_common_subsequence_length(pred_tokens, ref_tokens)

    if lcs_length == 0:
        return 0.0

    precision = lcs_length / len(pred_tokens)
    recall = lcs_length / len(ref_tokens)

    return 2 * precision * recall / (precision + recall)

In [47]:
SEMANTIC_MODEL_NAME = "intfloat/multilingual-e5-base"

semantic_model = SentenceTransformer(SEMANTIC_MODEL_NAME)


def compute_semantic_similarity(predictions: list, references: list) -> list:
    prefixed_predictions = ["query: " + text.strip() for text in predictions]
    prefixed_references = ["query: " + text.strip() for text in references]

    prediction_embeddings = semantic_model.encode(
        prefixed_predictions,
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    reference_embeddings = semantic_model.encode(
        prefixed_references,
        batch_size=16,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    similarities = (prediction_embeddings * reference_embeddings).sum(axis=1)

    return similarities.tolist()

## Per-Question and Aggregate Metrics

In [48]:
def evaluate_generations(records: list, retriever: str) -> pd.DataFrame:
    predictions = [record["generated_answer"] for record in records]
    references = [record["gold_answer"] for record in records]

    semantic_similarities = compute_semantic_similarity(predictions, references)

    rows = []

    for record, prediction, reference, semantic_sim in zip(
        records, predictions, references, semantic_similarities
    ):
        rows.append({
            "question_id": record["question_id"],
            "retriever": retriever,
            "EM": compute_exact_match(prediction, reference),
            "F1": compute_token_f1(prediction, reference),
            "ROUGE_L": compute_rouge_l(prediction, reference),
            "Semantic_sim": semantic_sim,
        })

    return pd.DataFrame(rows)


per_question_frames = [
    evaluate_generations(records, retriever)
    for retriever, records in generation_data.items()
]

per_question_df = pd.concat(per_question_frames, ignore_index=True)

per_question_df.head()

,question_id,retriever,EM,F1,ROUGE_L,Semantic_sim
0,61,tfidf,0,0.162162,0.162162,0.882360
1,25,tfidf,0,0.105263,0.084211,0.806694
2,27,tfidf,0,0.240000,0.120000,0.897165
3,130,tfidf,0,0.127660,0.085106,0.851178
4,33,tfidf,0,0.133333,0.133333,0.871563


In [49]:
metrics_df = (
    per_question_df
    .groupby("retriever")[["EM", "F1", "ROUGE_L", "Semantic_sim"]]
    .mean()
    .reset_index()
    .sort_values("F1", ascending=False)
)

metrics_df

,retriever,EM,F1,ROUGE_L,Semantic_sim
1,dense,0.0,0.171415,0.137277,0.856749
2,tfidf,0.0,0.149572,0.122563,0.861113
0,bm25,0.0,0.138575,0.112511,0.858987


## Comparing Retrievers

In [50]:
metrics_plot_df = metrics_df.melt(
    id_vars="retriever",
    value_vars=["EM", "F1", "ROUGE_L", "Semantic_sim"],
    var_name="metric",
    value_name="score",
)

alt.Chart(metrics_plot_df).mark_bar().encode(
    x=alt.X("retriever:N", title="Retriever"),
    y=alt.Y("score:Q", title="Score", scale=alt.Scale(domain=[0, 1])),
    color=alt.Color("metric:N", title="Metrika"),
    xOffset="metric:N",
    tooltip=["retriever", "metric", alt.Tooltip("score:Q", format=".3f")],
).properties(
    title=f"mT0-small — poređenje retrievera na {SPLIT} skupu",
    width=550,
    height=350,
)

alt.Chart(...)

## Inspecting Individual Answers

In [51]:
best_retriever = metrics_df.iloc[0]["retriever"]

sample_df = (
    per_question_df[per_question_df["retriever"] == best_retriever]
    .sort_values("F1")
    .head(5)
)

sample_records = {
    record["question_id"]: record
    for record in generation_data[best_retriever]
}

for _, row in sample_df.iterrows():
    record = sample_records[row["question_id"]]
    print(f"Pitanje: {record['question']}")
    print(f"Referentni odgovor: {record['gold_answer']}")
    print(f"Generisani odgovor: {record['generated_answer']}")
    print(f"EM={row['EM']:.0f}  F1={row['F1']:.3f}  ROUGE_L={row['ROUGE_L']:.3f}  Semantic_sim={row['Semantic_sim']:.3f}")
    print("-" * 80)

Pitanje: Navesti primere fatalnih posledica pri neispravnom softveru.
Referentni odgovor: Fatalne posledice mogu nastati kod kritičnih sistema. Therac-25 je zbog softverske greške doveo do najmanje šest slučajeva predoziranja pacijenata zračenjem, pri čemu su tri pacijenta umrla. Kod sistema Patriot greška u računanju vremena dovela je do promašaja od oko 600 metara, pri čemu raketa nije presretnuta, poginulo je 28 vojnika, a više od 100 je ranjeno. Raketa Ariane 5 se zbog softverske greške uništila 37 sekundi nakon poletanja, uz štetu veću od 370 miliona dolara.
Generisani odgovor: Uticaj neispravnog softvera u čak i kada žele da prijave problem i nemogu celovite podatke.
EM=0  F1=0.022  ROUGE_L=0.022  Semantic_sim=0.800
--------------------------------------------------------------------------------
Pitanje: Kakva je veza izvršivog koda i debagera?
Referentni odgovor: Debager mora da poveže stanje izvršivog programa sa odgovarajućim delovima izvornog koda i pomoćnim informacijama, ka

In [52]:
per_question_df.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_per_question_metrics.csv",
    index=False,
)

metrics_df_with_generator = metrics_df.copy()
metrics_df_with_generator.insert(0, "generator", GENERATOR_NAME)

metrics_df_with_generator.to_csv(
    RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metrics.csv",
    index=False,
)

metadata = {
    "generator": GENERATOR_NAME,
    "split": SPLIT,
    "rag_top_k": RAG_TOP_K,
    "retrievers_evaluated": list(generation_data.keys()),
    "metrics": ["EM", "F1", "ROUGE_L", "Semantic_sim"],
    "n_questions_per_retriever": {
        retriever: len(records)
        for retriever, records in generation_data.items()
    },
}

with (RESULTS_DIR / f"{GENERATOR_NAME}_{SPLIT}_metadata.json").open(
    "w", encoding="utf-8"
) as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

print("Sačuvani rezultati evaluacije u:", RESULTS_DIR)

Sačuvani rezultati evaluacije u: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/mt0_small


## mT0-small — zaključak

- Za razliku od prvog pokušaja, **mT0-small je uspešno naučio QA zadatak** — generisan tekst je gramatički razumljiv srpski.
- Poredak retrievera je konzistentan sa Qwen3-8B rezultatima i stabilan između validacije i testa: **dense > tfidf > bm25**, na obe metrike (F1, ROUGE-L). Ovo dodatno potvrđuje da je retrieval pipeline pouzdan nezavisno od generatora.
- Finalni test rezultati (dense, najbolji retriever): F1=0.171, ROUGE-L=0.137, Semantic_sim=0.857.
- Kvalitet generacije je osetno niži od Qwen3-8B. Ovo je očekivano: mT0-small ima red veličine manje parametara od Qwen3-8B i fine/tunovan je na svega ~100 primera, dok Qwen radi zero-shot oslanjajući se na ogromno pretrenirano znanje jezika.
- Preostala ograničenja uočena ručnim gledanjem: povremeno gramatički nepotpune rečenice, i sistematsko nepogađanje konkretnih imenovanih činjenica/primera iz referentnih odgovora — isti obrazac kao kod ostalih generatora, što sugeriše da je ovo delom i retrieval ograničenje (tačan chunk sa specifičnim imenima možda nije dosledno u top-k), ne samo ograničenje generatora.